# Gráficas del ejemplo 9: Fibonacci iterativo

Este notebook genera las gráficas estáticas de complejidad temporal $O(n)$ y espacial adicional $O(1)$.

In [ ]:
import importlib.util
from pathlib import Path
import sys
import time
import tracemalloc

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

_repo_root = next(
    base for base in (Path.cwd(), *Path.cwd().parents)
    if (base / "common").is_dir() and (base / "capitulo4").is_dir()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
_util_path = _repo_root / "capitulo4" / "runtime" / "util.py"
_spec = importlib.util.spec_from_file_location("capitulo4_reference_util", _util_path)
_util = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_util)
graficar_complejidad = _util.graficar_complejidad
modelo_lineal = _util.modelo_lineal
modelo_constante = _util.modelo_constante

In [ ]:
def fibonacci_iterativo(n):
    if n <= 1:
        return np.int32(n)

    a, b = np.int32(0), np.int32(1)
    with np.errstate(over="ignore"):
        for i in range(2, n + 1):
            c = a + b
            a = b
            b = c
    return b


def medir_tiempo(funcion, tamanos, ejecuciones):
    tiempos = np.zeros(len(tamanos), dtype=float)
    for indice, n in enumerate(tamanos):
        inicio = time.perf_counter()
        for _ in range(ejecuciones):
            funcion(int(n))
        tiempos[indice] = (time.perf_counter() - inicio) / ejecuciones
    return tiempos


def medir_memoria(funcion, tamanos, ejecuciones):
    memorias = np.zeros(len(tamanos), dtype=float)
    for indice, n in enumerate(tamanos):
        muestras = np.zeros(ejecuciones, dtype=float)
        tracemalloc.start()
        try:
            for repeticion in range(ejecuciones):
                tracemalloc.reset_peak()
                antes, _ = tracemalloc.get_traced_memory()
                resultado = funcion(int(n))
                _, pico = tracemalloc.get_traced_memory()
                muestras[repeticion] = max(0, pico - antes)
                del resultado
        finally:
            tracemalloc.stop()
        memorias[indice] = np.mean(muestras)
    return memorias

In [ ]:
ejecuciones = 50
tamanos = np.arange(1, 10_001, 200)

In [ ]:
tiempos = medir_tiempo(fibonacci_iterativo, tamanos, ejecuciones)
parametros_tiempo = curve_fit(modelo_lineal, tamanos, tiempos)[0]
tiempos_ajustados = modelo_lineal(tamanos, *parametros_tiempo)

graficar_complejidad(
    x=tamanos,
    y_experimental=tiempos,
    y_teorico=tiempos_ajustados,
    nombre_archivo="fibonacci_iterativo_tiempo.png",
    ylabel="Tiempo de ejecución [s]",
    funcion="T(n)",
)

In [ ]:
memorias = medir_memoria(fibonacci_iterativo, tamanos, ejecuciones)
parametros_memoria = curve_fit(modelo_constante, tamanos, memorias)[0]
memorias_ajustadas = modelo_constante(tamanos, *parametros_memoria)

graficar_complejidad(
    x=tamanos,
    y_experimental=memorias,
    y_teorico=memorias_ajustadas,
    nombre_archivo="fibonacci_iterativo_espacio.png",
    ylabel="Consumo de memoria [bytes]",
    funcion="S(n)",
)